# Spotify API ingestion pipeline


In [1]:
# --- Setup: Load environment and authenticate with Spotify API ---
import os
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyOAuth

# Load environment variables from .env
load_dotenv()

SPOTIFY_CLIENT_ID = os.getenv('SPOTIFY_CLIENT_ID')
SPOTIFY_CLIENT_SECRET = os.getenv('SPOTIFY_CLIENT_SECRET')
SPOTIFY_REDIRECT_URI = os.getenv('SPOTIFY_REDIRECT_URI', 'http://localhost:8888/callback')

# Check for required credentials
assert SPOTIFY_CLIENT_ID, 'Missing SPOTIFY_CLIENT_ID in .env'
assert SPOTIFY_CLIENT_SECRET, 'Missing SPOTIFY_CLIENT_SECRET in .env'

# Set up Spotipy client with user auth
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=SPOTIFY_CLIENT_ID,
    client_secret=SPOTIFY_CLIENT_SECRET,
    redirect_uri=SPOTIFY_REDIRECT_URI,
    scope='user-read-recently-played user-read-private',
    cache_path='.cache'
))

In [2]:
# Example: Get current user profile
user = sp.current_user()
print(f"Logged in as: {user['display_name']} ({user['id']})")

Logged in as: Uncle Ewan (1165229085)


In [3]:
# 1. Get the 50 most recent tracks
recently_played = sp.current_user_recently_played(limit=50)

# 2. Collect Artist IDs to fetch genres in one batch (more efficient)
artist_ids = [item['track']['artists'][0]['id'] for item in recently_played['items']]
# Get full artist objects (contains genres) for up to 50 IDs
full_artists = sp.artists(artist_ids)['artists']

# 3. Create a mapping of Artist ID -> Genres
genre_map = {art['id']: art['genres'] for art in full_artists}

# 4. Loop and print all your requested details
for item in recently_played['items']:
    track = item['track']
    
    # Required Data Points:
    name = track['name']
    artist_name = track['artists'][0]['name']
    artist_id = track['artists'][0]['id']
    played_at = item['played_at']  # This is the Time/Date
    genres = genre_map.get(artist_id, []) # Pull genre from our map
    
    print(f"Song: {name}")
    print(f"Artist: {artist_name}")
    print(f"Genre: {', '.join(genres) if genres else 'No Genre Data'}")
    print(f"Played At: {played_at}")
    print("-" * 30)

Song: Chewier
Artist: Ozric Tentacles
Genre: space rock, progressive rock, psychedelic rock, krautrock, jam band, neo-psychedelic, acid rock
Played At: 2026-04-05T11:35:48.414Z
------------------------------
Song: Bridge
Artist: Amon Tobin
Genre: idm, trip hop, nu jazz, downtempo
Played At: 2026-04-05T08:48:42.550Z
------------------------------
Song: Sploosh!
Artist: Ozric Tentacles
Genre: space rock, progressive rock, psychedelic rock, krautrock, jam band, neo-psychedelic, acid rock
Played At: 2026-04-05T08:42:42.355Z
------------------------------
Song: Sploosh!
Artist: Ozric Tentacles
Genre: space rock, progressive rock, psychedelic rock, krautrock, jam band, neo-psychedelic, acid rock
Played At: 2026-04-05T08:38:45.505Z
------------------------------
Song: Mysticum Arabicola
Artist: Ozric Tentacles
Genre: space rock, progressive rock, psychedelic rock, krautrock, jam band, neo-psychedelic, acid rock
Played At: 2026-04-04T14:47:37.200Z
------------------------------
Song: Sunscape
